In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Loading data and preliminary analysis


In [ ]:
!pip install -q pandas numpy torch matplotlib


In [ ]:
# Set paths
path_to_csv = '/content/drive/MyDrive/ML_DL_datasets/mentalhealth.csv'


In [ ]:
import re
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


In [ ]:
data = pd.read_csv(path_to_csv)


In [ ]:
data.head()


In [ ]:
# data preprocessing
for i in range(data.shape[0]):
    answer = str(data.at[i, 'Answers'])
    answer = re.sub(r'\n', ' ', answer)
    answer = re.sub(r'\(', '', answer)
    answer = re.sub(r'\)', '', answer)
    answer = re.sub(r',', '', answer)
    answer = re.sub(r'-', '', answer)
    answer = re.sub(r'/', '', answer)
    data.at[i, 'Answers'] = answer


In [ ]:
pairs = []

for i in range(data.shape[0]):
    pairs.append((data['Questions'][i], data['Answers'][i]))


In [ ]:
pairs  # questions


## Data preprocessing


In [ ]:
input_docs = []
target_docs = []
input_tokens = set()
target_tokens = set()

for line in pairs:
    input_doc, target_doc = line[0], line[1]

    # Appending each input sentence to input_docs
    input_docs.append(input_doc)

    # Splitting words from punctuation
    target_doc = " ".join(re.findall(r"[\w']+|[^\s\w]", target_doc))

    # Redefine target_doc below and append it to target_docs
    target_doc = '<START> ' + target_doc + ' <END>'

    target_docs.append(target_doc)

    for token in re.findall(r"[\w']+|[^\s\w]", input_doc):
        if token not in input_tokens:
            input_tokens.add(token)
    for token in target_doc.split():
        if token not in target_tokens:
            target_tokens.add(token)

input_tokens = sorted(list(input_tokens))  # contains all words of input_docs
target_tokens = sorted(list(target_tokens))
num_encoder_tokens = len(input_tokens)
num_decoder_tokens = len(target_tokens)


In [ ]:
input_docs


In [ ]:
target_docs


In [ ]:
input_features_dict = dict([(token, i) for i, token in enumerate(input_tokens)])
target_features_dict = dict([(token, i) for i, token in enumerate(target_tokens)])

reverse_target_features_dict = dict((i, token) for token, i in target_features_dict.items())


In [ ]:
input_features_dict


## Encoder - Decoder model


In [ ]:
max_encoder_seq_length = max([len(re.findall(r"[\w']+|[^\s\w]", input_doc)) for input_doc in input_docs])
max_decoder_seq_length = max([len(re.findall(r"[\w']+|[^\s\w]", target_doc)) for target_doc in target_docs])

encoder_input_data = np.zeros(
    (len(input_docs), max_encoder_seq_length, num_encoder_tokens),
    dtype='float32')
decoder_input_data = np.zeros(
    (len(input_docs), max_decoder_seq_length, num_decoder_tokens),
    dtype='float32')
decoder_target_data = np.full(
    (len(input_docs), max_decoder_seq_length),
    fill_value=-100,
    dtype='int64')

for line, (input_doc, target_doc) in enumerate(zip(input_docs, target_docs)):
    for timestep, token in enumerate(re.findall(r"[\w']+|[^\s\w]", input_doc)):
        # Assign 1. for the current line, timestep, & word in encoder_input_data
        encoder_input_data[line, timestep, input_features_dict[token]] = 1.

    for timestep, token in enumerate(target_doc.split()):
        decoder_input_data[line, timestep, target_features_dict[token]] = 1.
        if timestep > 0:
            decoder_target_data[line, timestep - 1] = target_features_dict[token]


In [ ]:
encoder_input_data


In [ ]:
decoder_target_data


## Training


In [ ]:
dimensionality = 256  # Dimensionality
batch_size = 10         # The batch size and number of epochs
epochs = 1000

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

class Seq2SeqLSTM(nn.Module):
    def __init__(self, num_encoder_tokens, num_decoder_tokens, hidden_dim):
        super().__init__()
        self.encoder_lstm = nn.LSTM(
            input_size=num_encoder_tokens,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.decoder_lstm = nn.LSTM(
            input_size=num_decoder_tokens,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.decoder_dense = nn.Linear(hidden_dim, num_decoder_tokens)

    def forward(self, encoder_inputs, decoder_inputs):
        _, (state_hidden, state_cell) = self.encoder_lstm(encoder_inputs)
        decoder_outputs, _ = self.decoder_lstm(decoder_inputs, (state_hidden, state_cell))
        decoder_outputs = self.decoder_dense(decoder_outputs)
        return decoder_outputs

    def encode(self, encoder_inputs):
        _, (state_hidden, state_cell) = self.encoder_lstm(encoder_inputs)
        return state_hidden, state_cell

    def decode_step(self, decoder_inputs, states):
        decoder_outputs, states = self.decoder_lstm(decoder_inputs, states)
        decoder_outputs = self.decoder_dense(decoder_outputs)
        return decoder_outputs, states


def token_accuracy(logits, targets, ignore_index=-100):
    with torch.no_grad():
        predictions = logits.argmax(dim=-1)
        valid_mask = targets != ignore_index
        if valid_mask.sum().item() == 0:
            return 0.0
        correct = (predictions == targets) & valid_mask
        return (correct.sum().float() / valid_mask.sum().float()).item()


In [ ]:
indices = np.arange(len(input_docs))
np.random.shuffle(indices)

if len(indices) < 2:
    train_indices = indices
    val_indices = indices
else:
    val_size = max(1, int(0.2 * len(indices)))
    if val_size >= len(indices):
        val_size = len(indices) - 1
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

encoder_tensor = torch.tensor(encoder_input_data, dtype=torch.float32)
decoder_input_tensor = torch.tensor(decoder_input_data, dtype=torch.float32)
decoder_target_tensor = torch.tensor(decoder_target_data, dtype=torch.long)

train_dataset = TensorDataset(
    encoder_tensor[train_indices],
    decoder_input_tensor[train_indices],
    decoder_target_tensor[train_indices]
)
val_dataset = TensorDataset(
    encoder_tensor[val_indices],
    decoder_input_tensor[val_indices],
    decoder_target_tensor[val_indices]
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

training_model = Seq2SeqLSTM(num_encoder_tokens, num_decoder_tokens, dimensionality).to(device)
optimizer = torch.optim.RMSprop(training_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)


In [ ]:
print(training_model)


In [ ]:
total_params = sum(p.numel() for p in training_model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")


In [ ]:
history1 = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

for epoch in range(epochs):
    training_model.train()
    running_loss = 0.0
    running_acc = 0.0

    for enc_batch, dec_in_batch, dec_tgt_batch in train_loader:
        enc_batch = enc_batch.to(device)
        dec_in_batch = dec_in_batch.to(device)
        dec_tgt_batch = dec_tgt_batch.to(device)

        optimizer.zero_grad()
        logits = training_model(enc_batch, dec_in_batch)
        loss = criterion(logits.reshape(-1, num_decoder_tokens), dec_tgt_batch.reshape(-1))
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += token_accuracy(logits, dec_tgt_batch)

    train_loss = running_loss / max(1, len(train_loader))
    train_acc = running_acc / max(1, len(train_loader))

    training_model.eval()
    val_loss_sum = 0.0
    val_acc_sum = 0.0

    with torch.no_grad():
        for enc_batch, dec_in_batch, dec_tgt_batch in val_loader:
            enc_batch = enc_batch.to(device)
            dec_in_batch = dec_in_batch.to(device)
            dec_tgt_batch = dec_tgt_batch.to(device)

            logits = training_model(enc_batch, dec_in_batch)
            loss = criterion(logits.reshape(-1, num_decoder_tokens), dec_tgt_batch.reshape(-1))

            val_loss_sum += loss.item()
            val_acc_sum += token_accuracy(logits, dec_tgt_batch)

    val_loss = val_loss_sum / max(1, len(val_loader))
    val_acc = val_acc_sum / max(1, len(val_loader))

    history1['accuracy'].append(train_acc)
    history1['val_accuracy'].append(val_acc)
    history1['loss'].append(train_loss)
    history1['val_loss'].append(val_loss)

    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"loss: {train_loss:.4f} - acc: {train_acc:.4f} - "
            f"val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
        )

torch.save(
    {
        'model_state_dict': training_model.state_dict(),
        'dimensionality': dimensionality,
        'num_encoder_tokens': num_encoder_tokens,
        'num_decoder_tokens': num_decoder_tokens,
        'input_features_dict': input_features_dict,
        'target_features_dict': target_features_dict,
        'reverse_target_features_dict': reverse_target_features_dict,
        'max_encoder_seq_length': max_encoder_seq_length,
        'max_decoder_seq_length': max_decoder_seq_length,
    },
    'training_model_pytorch.pt'
)


In [ ]:
acc = history1['accuracy']
val_acc = history1['val_accuracy']
loss = history1['loss']
val_loss = history1['val_loss']

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.show()


## Predictions


In [ ]:
# inference cho chatbot
# Encoder - sinh ra hidden states từ input
# Decoder - Dùng states vừa sinh ra + token trước để sinh ra token tiếp theo


In [ ]:
checkpoint = torch.load('training_model_pytorch.pt', map_location=device)

training_model = Seq2SeqLSTM(
    checkpoint['num_encoder_tokens'],
    checkpoint['num_decoder_tokens'],
    checkpoint['dimensionality']
).to(device)
training_model.load_state_dict(checkpoint['model_state_dict'])
training_model.eval()

input_features_dict = checkpoint['input_features_dict']
target_features_dict = checkpoint['target_features_dict']
reverse_target_features_dict = checkpoint['reverse_target_features_dict']
max_encoder_seq_length = checkpoint['max_encoder_seq_length']
max_decoder_seq_length = checkpoint['max_decoder_seq_length']
num_encoder_tokens = checkpoint['num_encoder_tokens']
num_decoder_tokens = checkpoint['num_decoder_tokens']


In [ ]:
def encoder_model(encoder_input_tensor):
    with torch.no_grad():
        return training_model.encode(encoder_input_tensor)


def decoder_model(decoder_input_tensor, decoder_states):
    with torch.no_grad():
        return training_model.decode_step(decoder_input_tensor, decoder_states)


In [ ]:
def decode_response(test_input):  # Hàm giải mã dùng để sinh câu trả lời với từng token một
    encoder_tensor = torch.tensor(test_input, dtype=torch.float32, device=device)

    # Timestep = 1
    states_value = encoder_model(encoder_tensor)

    target_seq = np.zeros((1, 1, num_decoder_tokens), dtype='float32')

    # Setting the first token of target sequence with the start token
    target_seq[0, 0, target_features_dict['<START>']] = 1.

    # A variable to store our response word by word
    decoded_sentence = ''

    stop_condition = False
    while not stop_condition:
        target_tensor = torch.tensor(target_seq, dtype=torch.float32, device=device)

        # Predicting output tokens with probabilities and states
        output_tokens, states_value = decoder_model(target_tensor, states_value)

        # Choosing the one with highest probability
        sampled_token_index = int(torch.argmax(output_tokens[0, -1, :]).item())
        sampled_token = reverse_target_features_dict[sampled_token_index]
        decoded_sentence += ' ' + sampled_token

        # Stop if hit max length or found the stop token
        if sampled_token == '<END>' or len(decoded_sentence.split()) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence
        target_seq = np.zeros((1, 1, num_decoder_tokens), dtype='float32')
        target_seq[0, 0, sampled_token_index] = 1.

    return decoded_sentence


In [ ]:
class ChatBot:
    negative_responses = ('no', 'nope', 'nah', 'naw', 'not a chance', 'sorry')
    exit_commands = ('quit', 'pause', 'exit', 'goodbye', 'bye', 'later', 'stop')

    # Method to start the conversation
    def start_chat(self):
        user_response = input("Hi, I'm a chatbot trained on random dialogs. AMA!\n")

        if user_response in self.negative_responses:
            print('Ok, have a great day!')
            return
        self.chat(user_response)

    # Method to handle the conversation
    def chat(self, reply):
        while not self.make_exit(reply):
            reply = input(self.generate_response(reply) + '\n')

    # Method to convert user input into a matrix
    def string_to_matrix(self, user_input):
        tokens = re.findall(r"[\w']+|[^\s\w]", user_input)
        user_input_matrix = np.zeros(
            (1, max_encoder_seq_length, num_encoder_tokens),
            dtype='float32'
        )
        for timestep, token in enumerate(tokens):
            if timestep < max_encoder_seq_length and token in input_features_dict:
                user_input_matrix[0, timestep, input_features_dict[token]] = 1.
        return user_input_matrix

    # Method that will create a response using seq2seq model we built
    def generate_response(self, user_input):
        input_matrix = self.string_to_matrix(user_input)
        chatbot_response = decode_response(input_matrix)

        # Remove <START> and <END> tokens from chatbot_response
        chatbot_response = chatbot_response.replace('<START>', '')
        chatbot_response = chatbot_response.replace('<END>', '')
        return chatbot_response

    # Method to check for exit commands
    def make_exit(self, reply):
        for exit_command in self.exit_commands:
            if exit_command in reply:
                print('Ok, have a great day!')
                return True
        return False


chatbot = ChatBot()


## Demo


In [ ]:
chatbot.start_chat()
